# Lab 04 · Federate MCP tools behind one endpoint

Publish the public Microsoft Learn MCP server through your AI Gateway, add a second backend under another namespace,
and call the federated tools with raw JSON-RPC and the MCP Python SDK. This notebook is the Python track of the
[lab guide](README.md): it creates the same `learn` tool server as the CLI track and runs the same checks.

**Register tool server → Add second backend → Handshake → List tools → Call a tool → Verify**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| Tool server `learn` (`toolServers`) | One governed MCP endpoint at `<gateway-url>/default/toolservers/learn/mcp`. |
| Backend `learn` (required) | The Microsoft Learn MCP server, `https://learn.microsoft.com/api/mcp`. Its tools appear as `learn_<tool>`. |
| Backend `docs` (optional) | The same server under a second namespace, so you can see federation without extra credentials. Its tools appear as `docs_<tool>`. |
| Runtime API key `app-first-call` | Authenticates the MCP client. The MCP endpoint accepts it only in the `api-key` header. |

![One MCP endpoint on the gateway federates tools from several backends](../assets/diagrams/mcp-federation.svg)

## Prerequisites

- An Azure subscription where you can create resources (Contributor on the resource group) and assign roles
  (Owner, User Access Administrator, or Role Based Access Control Administrator).
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  reuses your CLI sign-in through `AzureCliCredential`.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb) has run, so the gateway exists.
  Notebook 02 is optional: this notebook reuses its `app-first-call` key, or creates it.

Run the cells in order, or click **Run All**. Each step is idempotent: it reuses what already exists.

In [ ]:
%pip install -q --upgrade azure-identity requests httpx "mcp>=1.30,<2"

### 0. Initialize variables and clients

The defaults match notebooks 01 and 02. The tool server is named `learn`, like in the lab guide; lab 05 adds tool
controls to it.

In [ ]:
import hashlib, json, os, re, subprocess, sys, time
from pathlib import Path

import httpx
import requests
from azure.identity import AzureCliCredential
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

location       = os.environ.get("AIGW_LOCATION", "eastus2")                  # change to swedencentral if you prefer
resource_group = os.environ.get("AIGW_RESOURCE_GROUP", "rg-ai-gateway-labs") # change to use another resource group
gateway_name   = os.environ.get("AIGW_GATEWAY_NAME", "")                     # change to match notebook 01 (empty = derived)
api_key_name   = "app-first-call"                     # runtime API key from notebook 02
server_name    = "learn"                              # tool server name, part of the MCP endpoint URL
learn_mcp_url  = "https://learn.microsoft.com/api/mcp"

lab_name   = Path(globals().get("__vsc_ipynb_file__", "04-federate-mcp-tools.ipynb")).stem
lab_labels = {"lab": lab_name, "workload": "ai-gateway-labs"}   # tags for clean-up

result = subprocess.run("az account show -o json", shell=True, capture_output=True, text=True)
if result.returncode != 0:
    sys.exit("❌ Azure CLI not installed or not signed in. Run `az login` first.")
account         = json.loads(result.stdout)
subscription_id = account["id"]
tenant_id       = account["tenantId"]
user_name       = account["user"]["name"]

suffix       = hashlib.sha1(subscription_id.encode()).hexdigest()   # stable per subscription
gateway_name = gateway_name or f"aigw-labs-{suffix[:6]}"
assert location in ("eastus2", "swedencentral"), "During the preview, AI Gateway runs in eastus2 or swedencentral."
assert re.fullmatch(r"[A-Za-z]([A-Za-z0-9-]{0,22}[A-Za-z0-9])?", gateway_name), "Use 1-24 letters, digits, or hyphens."

ARM        = "https://management.azure.com"
ARM_API    = "2025-09-01-preview"   # Microsoft.ApiManagement/aigateways
GATEWAY_ID = (f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
              f"/providers/Microsoft.ApiManagement/aigateways/{gateway_name}")

credential = AzureCliCredential()   # every client below reuses your az login
_arm_token = None

def arm(method, path, body=None, api_version=ARM_API, params=None):
    """Call Azure Resource Manager and return the JSON body (None when a GET or DELETE finds nothing)."""
    global _arm_token
    if _arm_token is None or _arm_token.expires_on - time.time() < 300:
        _arm_token = credential.get_token("https://management.azure.com/.default")
    response = requests.request(method, ARM + path, json=body, timeout=120,
                                params={"api-version": api_version, **(params or {})},
                                headers={"Authorization": f"Bearer {_arm_token.token}"})
    if response.status_code == 404 and method in ("GET", "DELETE"):
        return None
    if not response.ok:
        raise RuntimeError(f"{method} {path} -> {response.status_code}: {response.text[:1000]}")
    return response.json() if response.content else {}

def wait_until_ready(path, api_version=ARM_API, label="Resource", timeout_minutes=30):
    """Poll properties.provisioningState until it reports Succeeded."""
    start = time.time()
    while True:
        resource = arm("GET", path, api_version=api_version)
        state = "NotFound" if resource is None else resource.get("properties", {}).get("provisioningState", "Succeeded")
        if state == "Succeeded":
            return resource
        if state in ("Failed", "Canceled"):
            raise RuntimeError(f"{label} provisioning {state}: {json.dumps(resource.get('properties'))[:1000]}")
        if time.time() - start > timeout_minutes * 60:
            raise TimeoutError(f"{label} is still {state} after {timeout_minutes} minutes.")
        print(f"⏳ {label}: {state} ({int(time.time() - start)} s)")
        time.sleep(20)

gateway = arm("GET", GATEWAY_ID)
if not gateway or gateway["properties"].get("provisioningState") != "Succeeded":
    sys.exit(f"❌ Gateway {gateway_name} not found in {resource_group}. Run 01-provision-gateway.ipynb first.")
gateway_url = "https://" + gateway["properties"]["frontend"]["defaultHostname"]
WS          = f"{GATEWAY_ID}/workspaces/default"   # models, aliases, tool servers, and policies live here

def get_or_create_key(name):
    """Create the runtime API key if it's missing, then read its primary value."""
    key_id = f"{GATEWAY_ID}/apiKeys/{name}"
    created = arm("GET", key_id) is None
    if created:
        arm("PUT", key_id, {"properties": {}})
    secrets = arm("POST", f"{key_id}/listSecrets")
    return secrets["primaryKey"], created

print(f"🧪 Lab:            {lab_name}")
print(f"👤 User:           {user_name}")
print(f"🏛️ Tenant:         {tenant_id}")
print(f"🔑 Subscription:   {account['name']} ({subscription_id})")
print(f"🌍 Region:         {location}")
print(f"📁 Resource group: {resource_group}")
print(f"🚪 Gateway:        {gateway_name}")
print(f"🌐 Runtime URL:    {gateway_url}")

### 1. Get a runtime API key

MCP clients authenticate to the gateway with the same runtime keys as model clients. The cell reuses `app-first-call`
from notebook 02, or creates it. It remembers whether it created the key, so clean-up removes only what this notebook
added.

In [ ]:
api_key, created_key = get_or_create_key(api_key_name)
print(("✅ Created" if created_key else "♻️ Reusing") + f" key {api_key_name}: {api_key[:4]}…")

### 2. Register the Learn MCP server

MCP servers are `toolServers` child resources of the gateway's `default` workspace. Each backend has a `namespace`
that prefixes its tool names, and `required: true` means the server fails if that backend is down. The gateway
publishes the MCP endpoint in `properties.gatewayEndpoints`.

In [ ]:
SERVER_PATH = f"{WS}/toolServers/{server_name}"

def learn_backend(namespace, required):
    return {"kind": "mcp", "namespace": namespace, "required": required,
            "mcp": {"transport": "streamableHttp", "url": learn_mcp_url}}

server = arm("GET", SERVER_PATH)
created_server = server is None
if created_server:
    arm("PUT", SERVER_PATH, {"properties": {
        "type": "mcp",
        "displayName": "Microsoft Learn",
        "description": "Search and fetch official Microsoft documentation and code samples.",
        "endpoints": [learn_backend("learn", True)],
    }})
    print(f"✅ Created tool server {server_name}")
else:
    print(f"♻️ Tool server {server_name} exists")

server  = wait_until_ready(SERVER_PATH, label="Tool server")
mcp_url = server["properties"]["gatewayEndpoints"][0]["url"]
print(f"🧰 MCP endpoint: {mcp_url}")
assert mcp_url.endswith(f"/default/toolservers/{server_name}/mcp"), mcp_url

### 3. Federate a second backend

Federation pays off when one endpoint combines several backends. To see it without extra credentials, add the same
Learn server again under the namespace `docs`, marked **not required**: if an optional backend is unavailable, the
gateway skips it instead of failing the whole server.

A `PATCH` of `properties.endpoints` replaces the whole list, so read the list, append the backend, and send the result
back.

In [ ]:
endpoints = arm("GET", SERVER_PATH)["properties"]["endpoints"]
if any(e.get("namespace") == "docs" for e in endpoints):
    print("♻️ Backend docs already federated")
else:
    endpoints = [e for e in endpoints if e.get("namespace") != "docs"] + [learn_backend("docs", False)]
    arm("PATCH", SERVER_PATH, {"properties": {"endpoints": endpoints}})
    print("✅ Added backend docs (not required)")

server = wait_until_ready(SERVER_PATH, label="Tool server")
for e in server["properties"]["endpoints"]:
    print(f"🔌 {e['namespace']:<6} required={e.get('required')!s:<5} {e['mcp']['url']}")

### 4. Run the MCP handshake and list tools

MCP clients start with a handshake: `initialize`, then `notifications/initialized`, then `tools/list`. Streamable HTTP
servers answer with JSON or server-sent events (SSE), so the helper reads both. After `initialize`, the client sends
the negotiated `MCP-Protocol-Version` and the `Mcp-Session-Id` the server returned. This is what
[infra/list-tools.sh](infra/list-tools.sh) does.

The new backend can take a minute to show up, so the cell retries until tools from both namespaces appear.

In [ ]:
def parse_mcp(response):
    """Return the JSON-RPC message from a JSON or SSE response."""
    if "text/event-stream" in response.headers.get("content-type", ""):
        data = [line[5:].strip() for line in response.text.splitlines() if line.startswith("data:")]
        return json.loads(data[-1]) if data else {}
    return response.json() if response.content else {}

def mcp_session(url, key):
    """Run initialize and notifications/initialized. Return the headers for later requests."""
    headers = {"api-key": key, "Content-Type": "application/json", "Accept": "application/json, text/event-stream"}
    init = requests.post(url, headers=headers, timeout=60, json={
        "jsonrpc": "2.0", "id": 1, "method": "initialize",
        "params": {"protocolVersion": "2025-03-26", "capabilities": {},
                   "clientInfo": {"name": "aigateway-labs", "version": "1.0"}}})
    init.raise_for_status()
    result = parse_mcp(init)["result"]
    headers["MCP-Protocol-Version"] = result["protocolVersion"]
    if init.headers.get("mcp-session-id"):
        headers["Mcp-Session-Id"] = init.headers["mcp-session-id"]
    requests.post(url, headers=headers, timeout=60, json={"jsonrpc": "2.0", "method": "notifications/initialized"})
    return headers, result

def mcp_request(url, headers, method, params=None, id=2):
    response = requests.post(url, headers=headers, timeout=120,
                             json={"jsonrpc": "2.0", "id": id, "method": method, "params": params or {}})
    response.raise_for_status()
    return parse_mcp(response)

for attempt in range(12):
    try:
        headers, server_info = mcp_session(mcp_url, api_key)
        tools = mcp_request(mcp_url, headers, "tools/list")["result"]["tools"]
    except (requests.HTTPError, KeyError) as error:
        print(f"⏳ {error}")
        tools = []
    namespaces = {t["name"].split("_", 1)[0] for t in tools}
    if {"learn", "docs"} <= namespaces:
        break
    print(f"⏳ Waiting for both backends ({sorted(namespaces)})")
    time.sleep(15)

print(f"🤝 Server: {server_info['serverInfo'].get('name')}  protocol: {server_info['protocolVersion']}\n")
for t in tools:
    print(f"🔧 {t['name']:<40} {' '.join((t.get('description') or '').split())[:60]}")
print(f"\n{len(tools)} tools published")
assert {"learn", "docs"} <= namespaces, "Tools from both namespaces should be listed."

### 5. Call a tool through the gateway

Call the Learn search tool by its namespaced name. Tool names are `<namespace>_<tool>`, so the cell picks the name
from the list instead of hard-coding it. The result is Microsoft Learn search results that came through your gateway.

In [ ]:
search_tool = next(t["name"] for t in tools if t["name"].startswith("learn_") and t["name"].endswith("docs_search"))
result = mcp_request(mcp_url, headers, "tools/call", id=3,
                     params={"name": search_tool, "arguments": {"query": "How do I add a token limit in AI Gateway?"}})
text = "\n".join(c.get("text", "") for c in result["result"]["content"] if c.get("type") == "text")
assert not result["result"].get("isError") and text

print(f"📨 {search_tool} returned:")
try:
    hits = json.loads(text)["results"]
    for hit in hits[:3]:
        print(f"📄 {hit['title']}  {hit.get('contentUrl', '')}")
    print(f"   {' '.join(hits[0]['content'].split())[:300]}…")
except (ValueError, KeyError, TypeError, IndexError):
    print(text[:600])

### 6. Connect with the MCP Python SDK

Real clients use an MCP SDK rather than raw JSON-RPC. The SDK runs the same handshake. Pass the runtime key in the
`api-key` header of the HTTP client; this is what [infra/list_tools.py](infra/list_tools.py) does. The cell uses
top-level `await`, which Jupyter supports.

In [ ]:
async with httpx.AsyncClient(headers={"api-key": api_key}, timeout=60) as http_client:
    async with streamable_http_client(mcp_url, http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session:
            init = await session.initialize()
            listed = await session.list_tools()

print(f"🤝 SDK connected to {init.serverInfo.name} (protocol {init.protocolVersion})")
print(f"🔧 {len(listed.tools)} tools: {', '.join(t.name for t in listed.tools)}")
assert len(listed.tools) == len(tools)

### 7. Verify that the gateway checks the caller

Microsoft Learn is public, but your MCP endpoint isn't. Send `tools/list` without a key: the gateway rejects it, and
no backend is called. The MCP endpoint reads the key only from the `api-key` header, so a bearer runtime key is
rejected too.

In [ ]:
probe = {"jsonrpc": "2.0", "id": 1, "method": "tools/list"}
accept = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}

no_key = requests.post(mcp_url, json=probe, headers=accept, timeout=60)
print(f"📨 Without a key:       HTTP {no_key.status_code}")
assert no_key.status_code == 401

bearer = requests.post(mcp_url, json=probe, headers={**accept, "Authorization": f"Bearer {api_key}"}, timeout=60)
print(f"📨 Key as bearer token: HTTP {bearer.status_code}")
assert bearer.status_code == 401
print("✅ The MCP endpoint only accepts callers with a key in the api-key header")

### 8. Clean up resources

Lab 05 adds tool controls to the `learn` server, so the cell keeps a server that existed before this notebook ran and
removes only the `docs` backend. It deletes the server and the key only if this notebook created them. Set
`keep_for_lab_05 = True` to keep everything.

In [ ]:
keep_for_lab_05 = False   # change to True to keep the tool server for lab 05

if keep_for_lab_05:
    print(f"ℹ️ Kept tool server {server_name} and key {api_key_name}")
else:
    if created_server:
        arm("DELETE", SERVER_PATH)
        print(f"🗑️ Deleted tool server {server_name}")
    else:
        endpoints = [e for e in arm("GET", SERVER_PATH)["properties"]["endpoints"] if e.get("namespace") != "docs"]
        arm("PATCH", SERVER_PATH, {"properties": {"endpoints": endpoints}})
        print(f"🗑️ Removed backend docs from {server_name}")
    if created_key:
        arm("DELETE", f"{GATEWAY_ID}/apiKeys/{api_key_name}")
        print(f"🗑️ Deleted key {api_key_name}")

## Troubleshooting

- **`401` from the MCP endpoint with a key**: send the key in the `api-key` header. `Authorization: Bearer <key>`
  works for models but not for MCP. A new key can take a minute to work.
- **Only `learn_` tools are listed**: the `docs` backend isn't applied yet, or the `PATCH` dropped it. Rerun step 3
  and check `arm("GET", SERVER_PATH)["properties"]["endpoints"]`.
- **JSON-RPC `error` instead of a result**: the server needs the full handshake. Send `initialize` and
  `notifications/initialized` first, and pass `MCP-Protocol-Version` and `Mcp-Session-Id` on later requests.
- **`406 Not Acceptable`**: include `Accept: application/json, text/event-stream`.
- **`Unknown tool`**: tool names carry the namespace prefix, for example `learn_microsoft_docs_search`. Use the names
  from `tools/list`.
- **`ImportError: streamable_http_client`**: install `mcp>=1.30,<2`, then restart the kernel.

## Next steps

- Control which tools agents can call in [lab 05](../05-mcp-tool-controls/README.md). Set `keep_for_lab_05 = True` in step 8 first.
- Throttle model callers with [token limits](../06-token-limits/README.md) and its notebook [06-token-limits.ipynb](../06-token-limits/06-token-limits.ipynb).
- Docs: [Connect to MCP tools through AI Gateway](https://ai.gateway.azure.com/docs/mcp-access).
- Read this lab on the labs site: [https://azure.github.io/ai-gateway/labs/04-federate-mcp-tools/](https://azure.github.io/ai-gateway/labs/04-federate-mcp-tools/).